# 🏦 Loan Approval 2025 — Dataset Analysis & Model Comparison
**Classification Type:** Binary (Approved / Rejected)  
**Goal:** Address preprocessing issues, train two models, and evaluate performance using Accuracy, Precision, Recall, F1, and ROC AUC  
**Models:** Random Forest vs. XGBoost  

**Preprocessing Issues Addressed:**
- ✅ Outliers in `savings_assets`, `annual_income`, `current_debt` → log transformation
- ✅ Multicollinearity in ratio features → VIF check + redundant feature removal
- ✅ Skewed binary features (`defaults_on_file`, `derogatory_marks`) → kept as strong signals
- ✅ Categorical encoding → One-Hot Encoding (fit on train only)
- ✅ Stratified train/test split to preserve sub-group representation

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Loan_approval_data_2025.csv')

# Drop ID column — not a predictive feature
df = df.drop(columns=['customer_id'])

print('Shape:', df.shape)
df.head()

## 3. Basic Info & Null Check

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nNull Values per column:')
print(df.isnull().sum())
print(f'\nTotal null values: {df.isnull().sum().sum()}')
print('\nUnique values per column:')
print(df.nunique())

## 4. Class Balance Check

In [ ]:
print('Class Distribution:')
print(df['loan_status'].value_counts())
print('\n% Distribution:')
print(df['loan_status'].value_counts(normalize=True) * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['loan_status'].value_counts().sort_index()
colors = ['#e74c3c', '#2ecc71']
labels = ['Rejected (0)', 'Approved (1)']

axes[0].bar(labels, counts.values, color=colors, edgecolor='black')
axes[0].set_title('Class Distribution — Count')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Class Distribution — Proportion')

plt.suptitle('🏦 Loan Approval 2025 — Class Balance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n✅ Dataset is BALANCED (~45% Rejected / ~55% Approved) — ratio 1.22:1')
print('   No SMOTE or class weighting required.')

## 5. Exploratory Data Analysis

In [ ]:
# Approval rates by categorical features
cat_cols = ['occupation_status', 'product_type', 'loan_intent']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, cat_cols):
    approval_rate = df.groupby(col)['loan_status'].mean().sort_values(ascending=False) * 100
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(approval_rate)))
    approval_rate.plot(kind='bar', ax=ax, color=colors[::-1], edgecolor='black')
    ax.set_title(f'Approval Rate by {col}', fontweight='bold')
    ax.set_ylabel('Approval Rate (%)')
    ax.set_ylim(0, 85)
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.suptitle('🏦 Approval Rates by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Key numeric feature distributions by loan status
key_features = ['credit_score', 'annual_income', 'debt_to_income_ratio',
                'savings_assets', 'loan_amount', 'interest_rate']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    for status, color, label in [(0, '#e74c3c', 'Rejected'), (1, '#2ecc71', 'Approved')]:
        subset = df[df['loan_status'] == status][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=color, label=label, edgecolor='none')
    ax.set_title(feat, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.suptitle('🏦 Feature Distributions by Loan Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Issue #1 — Outliers & Skewed Features: Log Transformation

> **Why:** `savings_assets` has 13% IQR outliers and a mean (3,596) far below its max (300,000).  
> `annual_income` and `current_debt` are similarly right-skewed.  
> Log transformation compresses extreme values without removing data, and brings distributions closer to normal — improving performance for distance-sensitive and linear components of the pipeline.
> We use `log1p` (i.e. log(1+x)) to safely handle zero values.

In [ ]:
log_cols = ['annual_income', 'savings_assets', 'current_debt', 'loan_amount']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for i, col in enumerate(log_cols):
    # Before
    axes[0, i].hist(df[col], bins=50, color='#e74c3c', edgecolor='none', alpha=0.8)
    axes[0, i].set_title(f'{col}\n(before)', fontweight='bold')
    axes[0, i].set_ylabel('Frequency')

    # After
    axes[1, i].hist(np.log1p(df[col]), bins=50, color='#2ecc71', edgecolor='none', alpha=0.8)
    axes[1, i].set_title(f'log1p({col})\n(after)', fontweight='bold')
    axes[1, i].set_ylabel('Frequency')

plt.suptitle('🏦 Log Transformation: Before vs After', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Apply to dataset
df_processed = df.copy()
for col in log_cols:
    df_processed[col] = np.log1p(df_processed[col])

print('✅ Log1p transformation applied to:', log_cols)

## 7. Issue #2 — Multicollinearity: VIF Check on Ratio Features

> **Why:** `debt_to_income_ratio`, `loan_to_income_ratio`, and `payment_to_income_ratio` are all derived from the same underlying fields (income + debt + loan).  
> High multicollinearity inflates coefficient variance in linear models and can confuse feature importance scores.  
> We use Variance Inflation Factor (VIF) to quantify this — VIF > 10 signals a problematic feature.

In [ ]:
ratio_cols = ['debt_to_income_ratio', 'loan_to_income_ratio', 'payment_to_income_ratio']
ratio_data = df_processed[ratio_cols].copy()

vif_data = pd.DataFrame()
vif_data['Feature'] = ratio_cols
vif_data['VIF'] = [variance_inflation_factor(ratio_data.values, i)
                   for i in range(ratio_data.shape[1])]

print('=== VIF Scores for Ratio Features ===')
print(vif_data.to_string(index=False))
print('\nVIF > 10 = high multicollinearity | VIF > 5 = moderate')

# Visualise correlation between ratio features
fig, ax = plt.subplots(figsize=(6, 4))
corr = df_processed[ratio_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            square=True, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Correlation Between Ratio Features', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Drop the most redundant ratio feature based on VIF
# payment_to_income_ratio is the most derivable from the other two — drop it
df_processed = df_processed.drop(columns=['payment_to_income_ratio'])

print('✅ Dropped payment_to_income_ratio (highest redundancy with other ratio features).')
print(f'   Remaining features: {df_processed.shape[1] - 1} (excl. target)')

## 8. Issue #3 — Categorical Encoding

> **Why:** Machine learning models require numeric inputs. We use One-Hot Encoding for all three categorical features.  
> Encoders are fit **only on the training set** to prevent data leakage.
> `drop='first'` removes one dummy per category to avoid the dummy variable trap.

In [ ]:
cat_cols = ['occupation_status', 'product_type', 'loan_intent']

print('Categorical columns to encode:', cat_cols)
for col in cat_cols:
    print(f'  {col}: {df_processed[col].unique().tolist()}')

## 9. Train / Test Split (Stratified)

> **Why stratified:** Even though the overall dataset is balanced (45/55), sub-groups like loan_intent are not  
> (Debt Consolidation: 36.6% approval vs Education: 67.5%). Stratification ensures these sub-groups are  
> proportionally represented in both train and test sets.

In [ ]:
X_raw = df_processed.drop(columns=['loan_status'])
y = df_processed['loan_status']

# Stratified split BEFORE encoding (prevents leakage)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set:  {X_train_raw.shape[0]:,} samples')
print(f'Test set:      {X_test_raw.shape[0]:,} samples')
print(f'\nTrain class balance: {y_train.value_counts(normalize=True).mul(100).round(1).to_dict()}')
print(f'Test class balance:  {y_test.value_counts(normalize=True).mul(100).round(1).to_dict()}')
print('\n✅ Stratified split preserves class proportions in both sets.')

In [ ]:
# One-Hot Encode categorical columns — fit on train only
X_train_enc = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True)
X_test_enc  = pd.get_dummies(X_test_raw,  columns=cat_cols, drop_first=True)

# Align test set columns to training set (handles any unseen categories)
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print(f'Features after encoding: {X_train_enc.shape[1]}')
print('\nFinal feature list:')
print(X_train_enc.columns.tolist())
print('\n✅ One-Hot Encoding fit on training data only — no leakage.')

## 10. Feature Scaling

> **Note:** XGBoost and Random Forest are tree-based models — they are scale-invariant and do not require feature scaling.  
> We scale anyway to maintain a clean, reusable pipeline and for consistency if other models are added later.  
> Scaler is fit on training set only.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)

# Convert back to DataFrame for model-readability
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_enc.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test_enc.columns)

print('✅ StandardScaler fit on training data only.')
print(f'   Train mean (should be ~0): {X_train_scaled.mean().mean():.4f}')
print(f'   Train std  (should be ~1): {X_train_scaled.std().mean():.4f}')

## 11. Model Training & Evaluation

### 11A. Model 1 — Random Forest

In [ ]:
def evaluate_model(y_test, y_pred, y_prob, label='Model'):
    print(f'=== {label} ===')
    print(f'Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision: {precision_score(y_test, y_pred):.4f}')
    print(f'Recall:    {recall_score(y_test, y_pred):.4f}')
    print(f'F1 Score:  {f1_score(y_test, y_pred):.4f}')
    print(f'ROC AUC:   {roc_auc_score(y_test, y_prob):.4f}')
    print()
    return {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1 Score':  f1_score(y_test, y_pred),
        'ROC AUC':   roc_auc_score(y_test, y_prob)
    }

# Random Forest — regularised to fix overfitting (was: max_depth=None, min_samples_split=5)
# max_depth=10    : prevents trees growing until every sample is perfectly classified
# min_samples_split=10 : requires more samples before making a split
# min_samples_leaf=4   : each leaf must contain at least 4 samples
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,           # was None — key fix for overfitting
    min_samples_split=10,   # was 5
    min_samples_leaf=4,     # new
    max_features='sqrt',    # explicit default
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_enc, y_train)
y_pred_rf = rf.predict(X_test_enc)
y_prob_rf  = rf.predict_proba(X_test_enc)[:, 1]
rf_scores  = evaluate_model(y_test, y_pred_rf, y_prob_rf, 'Random Forest')

### 11A-i. Random Forest — `max_depth` Sweep

> Find the optimal `max_depth` by plotting Train vs. Test F1 across a range of depths.  
> The sweet spot is where **Test F1 peaks** before the train/test gap widens again.

In [ ]:
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
import numpy as np

depths = [3, 5, 7, 10, 12, 15, 20, None]
train_f1s, test_f1s, gaps = [], [], []

print(f"{'max_depth':<12} {'Train F1':>10} {'Test F1':>10} {'Gap':>8}")
print('-' * 44)

for d in depths:
    rf_temp = RandomForestClassifier(
        n_estimators=200, max_depth=d,
        min_samples_split=10, min_samples_leaf=4,
        max_features='sqrt', random_state=42, n_jobs=-1
    )
    rf_temp.fit(X_train_enc, y_train)
    tr = f1_score(y_train, rf_temp.predict(X_train_enc))
    te = f1_score(y_test,  rf_temp.predict(X_test_enc))
    train_f1s.append(tr)
    test_f1s.append(te)
    gaps.append(tr - te)
    label = str(d) if d is not None else 'None'
    marker = ' ← current' if d == 10 else ''
    print(f"{label:<12} {tr:>10.4f} {te:>10.4f} {tr-te:>+8.4f}{marker}")

# Identify best depth
best_idx  = int(np.argmax(test_f1s))
best_depth = depths[best_idx]
print(f"\n🏆 Best max_depth by Test F1: {best_depth} ({test_f1s[best_idx]:.4f})")

# Plot
depth_labels = [str(d) if d is not None else 'None' for d in depths]
x = np.arange(len(depths))

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

ax1.plot(x, train_f1s, 'o-', color='#2ecc71', linewidth=2, label='Train F1')
ax1.plot(x, test_f1s,  's-', color='steelblue', linewidth=2, label='Test F1')
ax2.bar(x, gaps, alpha=0.25, color='#e74c3c', label='Gap (Train−Test)')

ax1.axvline(x=best_idx, color='gold', linestyle='--', linewidth=1.5,
            label=f'Best depth = {best_depth}')
ax1.axvline(x=depths.index(10), color='gray', linestyle=':', linewidth=1.2,
            label='Current depth = 10')

ax1.set_xticks(x)
ax1.set_xticklabels(depth_labels)
ax1.set_xlabel('max_depth')
ax1.set_ylabel('F1 Score')
ax2.set_ylabel('Train−Test Gap', color='#e74c3c')
ax1.set_title('Random Forest — max_depth Sweep (Train vs. Test F1)', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)
plt.tight_layout()
plt.show()

# Store best depth for retraining below
rf_best_depth = best_depth

### 11A-ii. Random Forest — Retrain with Optimal `max_depth`

In [ ]:
# Retrain RF with the optimal max_depth found from the sweep
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=rf_best_depth,     # optimised from sweep above
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_enc, y_train)
y_pred_rf = rf.predict(X_test_enc)
y_prob_rf  = rf.predict_proba(X_test_enc)[:, 1]
rf_scores  = evaluate_model(y_test, y_pred_rf, y_prob_rf,
                             f'Random Forest (max_depth={rf_best_depth})')
print(f'✅ RF retrained with max_depth={rf_best_depth} (from sweep).')

### 11B. Model 2 — XGBoost

In [ ]:
# XGBoost — regularised to reduce mild overfitting
# max_depth reduced 6→4, learning_rate slowed 0.1→0.05
# subsample/colsample tightened, L1/L2 regularisation added
# use_label_encoder removed (deprecated since XGBoost 1.6)
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,            # was 6 — shallower trees
    learning_rate=0.05,     # was 0.1 — slower learning = better generalisation
    subsample=0.7,          # was 0.8
    colsample_bytree=0.7,   # was 0.8
    reg_alpha=0.1,          # new — L1 regularisation
    reg_lambda=1.5,         # new — L2 regularisation (default is 1)
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_enc, y_train)
y_pred_xgb = xgb_model.predict(X_test_enc)
y_prob_xgb  = xgb_model.predict_proba(X_test_enc)[:, 1]
xgb_scores  = evaluate_model(y_test, y_pred_xgb, y_prob_xgb, 'XGBoost')

### 11B-i. XGBoost — Hyperparameter Tuning (`RandomizedSearchCV`)

> `RandomizedSearchCV` samples random combinations from the param grid —  
> faster than `GridSearchCV` while still covering a wide search space.  
> We use 5-fold stratified CV scored on F1 to find the best combination.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'max_depth':        randint(3, 9),           # 3–8
    'learning_rate':    uniform(0.01, 0.19),      # 0.01–0.20
    'n_estimators':     randint(100, 401),        # 100–400
    'subsample':        uniform(0.6, 0.4),        # 0.6–1.0
    'colsample_bytree': uniform(0.6, 0.4),        # 0.6–1.0
    'reg_alpha':        uniform(0.0, 0.5),        # 0.0–0.5
    'reg_lambda':       uniform(1.0, 2.0),        # 1.0–3.0
}

xgb_base = xgb.XGBClassifier(
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=40,                        # 40 random combinations
    scoring='f1',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train_enc, y_train)

print('\n🏆 Best CV F1:', round(search.best_score_, 4))
print('Best params:')
for k, v in search.best_params_.items():
    print(f'  {k:<20} {v}')

xgb_best_params = search.best_params_

### 11B-ii. XGBoost — Retrain with Best Params

In [ ]:
# Retrain XGBoost with the best hyperparameters found by RandomizedSearchCV
xgb_model = xgb.XGBClassifier(
    **xgb_best_params,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_enc, y_train)
y_pred_xgb = xgb_model.predict(X_test_enc)
y_prob_xgb  = xgb_model.predict_proba(X_test_enc)[:, 1]
xgb_scores  = evaluate_model(y_test, y_pred_xgb, y_prob_xgb, 'XGBoost (tuned)')
print('✅ XGBoost retrained with RandomizedSearchCV best params.')

## 12. Classification Reports

In [ ]:
print('Classification Report — Random Forest:')
print(classification_report(y_test, y_pred_rf, target_names=['Rejected', 'Approved']))

print('Classification Report — XGBoost:')
print(classification_report(y_test, y_pred_xgb, target_names=['Rejected', 'Approved']))

## 13. Confusion Matrices — Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                       display_labels=['Rejected', 'Approved']).plot(
    ax=axes[0], colorbar=False, cmap='Greens'
)
axes[0].set_title('Random Forest\nConfusion Matrix', fontweight='bold')

cm_xgb = confusion_matrix(y_test, y_pred_xgb)
ConfusionMatrixDisplay(confusion_matrix=cm_xgb,
                       display_labels=['Rejected', 'Approved']).plot(
    ax=axes[1], colorbar=False, cmap='Blues'
)
axes[1].set_title('XGBoost\nConfusion Matrix', fontweight='bold')

plt.suptitle('🏦 Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 14. ROC Curves — Side by Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_prob_rf, ax=axes[0], color='#2ecc71')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random Baseline')
axes[0].set_title('Random Forest — ROC Curve', fontweight='bold')
axes[0].legend()

RocCurveDisplay.from_predictions(y_test, y_prob_xgb, ax=axes[1], color='steelblue')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Baseline')
axes[1].set_title('XGBoost — ROC Curve', fontweight='bold')
axes[1].legend()

plt.suptitle('🏦 ROC Curve Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 15. Overlaid ROC Curves (Both Models on One Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(y_test, y_prob_rf,  ax=ax, color='#2ecc71', name='Random Forest')
RocCurveDisplay.from_predictions(y_test, y_prob_xgb, ax=ax, color='steelblue', name='XGBoost')
ax.plot([0, 1], [0, 1], 'k--', label='Random Baseline')
ax.set_title('🏦 ROC Curves — Random Forest vs XGBoost', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 16. Feature Importance

### 16A. Random Forest — Feature Importance Scores

In [ ]:
importances_rf = pd.Series(rf.feature_importances_, index=X_train_enc.columns)
top15_rf = importances_rf.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top15_rf.sort_values().plot(kind='barh', color='#2ecc71', edgecolor='black')
plt.title('Random Forest — Top 15 Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 15 Features (Random Forest):')
print(top15_rf)

### 16B. XGBoost — Feature Importance Scores

In [ ]:
importances_xgb = pd.Series(xgb_model.feature_importances_, index=X_train_enc.columns)
top15_xgb = importances_xgb.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top15_xgb.sort_values().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('XGBoost — Top 15 Feature Importances', fontsize=13, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 15 Features (XGBoost):')
print(top15_xgb)

### 16C. Feature Importance Comparison — Side by Side

In [ ]:
# Top 10 from each model side by side
top10_rf  = importances_rf.sort_values(ascending=False).head(10)
top10_xgb = importances_xgb.sort_values(ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top10_rf.sort_values().plot(kind='barh', ax=axes[0], color='#2ecc71', edgecolor='black')
axes[0].set_title('Random Forest — Top 10 Features', fontweight='bold')
axes[0].set_xlabel('Importance Score')

top10_xgb.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('XGBoost — Top 10 Features', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.suptitle('🏦 Feature Importance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 17. Head-to-Head Model Comparison

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']

comparison_df = pd.DataFrame({
    'Random Forest': [rf_scores[m]  for m in metrics],
    'XGBoost':       [xgb_scores[m] for m in metrics]
}, index=metrics)

print('=== Head-to-Head Model Comparison ===')
print(comparison_df.round(4).to_string())
print()

# Highlight which model wins each metric
print('=== Winner per Metric ===')
for metric in metrics:
    rf_val  = rf_scores[metric]
    xgb_val = xgb_scores[metric]
    if rf_val > xgb_val:
        print(f'{metric}: Random Forest wins ({rf_val:.4f} vs {xgb_val:.4f})')
    elif xgb_val > rf_val:
        print(f'{metric}: XGBoost wins ({xgb_val:.4f} vs {rf_val:.4f})')
    else:
        print(f'{metric}: Tie ({rf_val:.4f})')

In [ ]:
# Bar chart comparison
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width/2, comparison_df['Random Forest'], width,
               label='Random Forest', color='#2ecc71', edgecolor='black')
bars2 = ax.bar(x + width/2, comparison_df['XGBoost'], width,
               label='XGBoost', color='steelblue', edgecolor='black')

# Dynamic y-axis — zoom in to show differences clearly
all_vals = list(comparison_df['Random Forest']) + list(comparison_df['XGBoost'])
y_min = max(0, min(all_vals) - 0.02)
ax.set_ylim(y_min, 1.01)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylabel('Score')
ax.set_title('🏦 Random Forest vs. XGBoost — All Metrics', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 18. Cross-Validation — Robustness Check (Both Models)

In [ ]:
# Prepare full encoded dataset for CV
X_full = pd.get_dummies(X_raw, columns=cat_cols, drop_first=True)
y_full = y

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Random Forest CV
cv_rf = cross_val_score(
    RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=10, min_samples_leaf=4, max_features='sqrt', random_state=42, n_jobs=-1),
    X_full, y_full, cv=skf, scoring='f1'
)
print('Random Forest — 5-Fold Stratified CV F1 Scores:', np.round(cv_rf, 4))
print(f'Mean F1: {cv_rf.mean():.4f} (+/- {cv_rf.std():.4f})')
print()

# XGBoost CV
cv_xgb = cross_val_score(
    xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                      subsample=0.7, colsample_bytree=0.7,
                      reg_alpha=0.1, reg_lambda=1.5,
                      eval_metric='logloss', random_state=42, n_jobs=-1),
    X_full, y_full, cv=skf, scoring='f1'
)
print('XGBoost — 5-Fold Stratified CV F1 Scores:', np.round(cv_xgb, 4))
print(f'Mean F1: {cv_xgb.mean():.4f} (+/- {cv_xgb.std():.4f})')

print('\n✅ Stratified CV ensures all loan_intent and product_type sub-groups are represented in every fold.')

In [ ]:
# CV comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))

folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)
width = 0.35

ax.bar(x - width/2, cv_rf,  width, label='Random Forest', color='#2ecc71', edgecolor='black')
ax.bar(x + width/2, cv_xgb, width, label='XGBoost',       color='steelblue', edgecolor='black')

all_cv = list(cv_rf) + list(cv_xgb)
ax.set_ylim(max(0, min(all_cv) - 0.02), 1.01)
ax.axhline(cv_rf.mean(),  color='#2ecc71',  linestyle='--', linewidth=1.2, label=f'RF Mean: {cv_rf.mean():.4f}')
ax.axhline(cv_xgb.mean(), color='steelblue', linestyle='--', linewidth=1.2, label=f'XGB Mean: {cv_xgb.mean():.4f}')

ax.set_xticks(x)
ax.set_xticklabels(folds)
ax.set_ylabel('F1 Score')
ax.set_title('🏦 5-Fold CV F1 Scores — Random Forest vs XGBoost', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 19. Train vs. Test Performance — Overfitting Diagnostic

> **Why this matters:** A model that scores perfectly on training data but poorly on test data is overfitting (memorising, not learning).  
> A model that scores poorly on *both* is underfitting.  
> We want a **small gap** between train and test scores — ideally < 0.02 on F1.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

# ── Train predictions ────────────────────────────────────────────────────────
y_train_pred_rf  = rf.predict(X_train_enc)
y_train_prob_rf  = rf.predict_proba(X_train_enc)[:, 1]

y_train_pred_xgb = xgb_model.predict(X_train_enc)
y_train_prob_xgb = xgb_model.predict_proba(X_train_enc)[:, 1]

# ── Compute metrics ──────────────────────────────────────────────────────────
metrics_list = ['Accuracy', 'F1 Score', 'ROC AUC']

def get_scores(y_true, y_pred, y_prob):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1 Score': f1_score(y_true, y_pred),
        'ROC AUC':  roc_auc_score(y_true, y_prob)
    }

rf_train_scores  = get_scores(y_train, y_train_pred_rf,  y_train_prob_rf)
rf_test_scores   = get_scores(y_test,  y_pred_rf,        y_prob_rf)

xgb_train_scores = get_scores(y_train, y_train_pred_xgb, y_train_prob_xgb)
xgb_test_scores  = get_scores(y_test,  y_pred_xgb,       y_prob_xgb)

# ── Print summary table ───────────────────────────────────────────────────────
print('=' * 65)
print(f"{'Metric':<12} {'RF Train':>10} {'RF Test':>10} {'Gap':>8}  │  {'XGB Train':>10} {'XGB Test':>10} {'Gap':>8}")
print('=' * 65)
for m in metrics_list:
    rf_gap  = rf_train_scores[m]  - rf_test_scores[m]
    xgb_gap = xgb_train_scores[m] - xgb_test_scores[m]
    print(f"{m:<12} {rf_train_scores[m]:>10.4f} {rf_test_scores[m]:>10.4f} {rf_gap:>+8.4f}  │  "
          f"{xgb_train_scores[m]:>10.4f} {xgb_test_scores[m]:>10.4f} {xgb_gap:>+8.4f}")
print('=' * 65)
print()
print('Gap = Train − Test  |  Positive gap = model scores higher on training data')
print('Rule of thumb: Gap < 0.02 → well-generalised | > 0.05 → likely overfitting')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

x = np.arange(2)          # Train | Test
tick_labels = ['Train', 'Test']
width = 0.35

colors_rf  = ['#2ecc71', '#27ae60']
colors_xgb = ['#3498db', '#1a6fa8']

for i, metric in enumerate(metrics_list):
    ax = axes[i]

    rf_vals  = [rf_train_scores[metric],  rf_test_scores[metric]]
    xgb_vals = [xgb_train_scores[metric], xgb_test_scores[metric]]

    bars_rf  = ax.bar(x - width/2, rf_vals,  width, label='Random Forest',
                      color=colors_rf,  edgecolor='black', alpha=0.9)
    bars_xgb = ax.bar(x + width/2, xgb_vals, width, label='XGBoost',
                      color=colors_xgb, edgecolor='black', alpha=0.9)

    # Annotate bar values
    for bar in list(bars_rf) + list(bars_xgb):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.002,
                f'{bar.get_height():.4f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')

    # Shade the gap region for RF
    rf_gap  = rf_train_scores[metric]  - rf_test_scores[metric]
    xgb_gap = xgb_train_scores[metric] - xgb_test_scores[metric]

    all_vals = rf_vals + xgb_vals
    y_min = max(0, min(all_vals) - 0.03)
    ax.set_ylim(y_min, min(1.03, max(all_vals) + 0.04))
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, fontsize=10)
    ax.set_title(f'{metric}\nRF gap: {rf_gap:+.4f}  |  XGB gap: {xgb_gap:+.4f}',
                 fontsize=10, fontweight='bold')
    ax.set_ylabel('Score')
    if i == 0:
        ax.legend(fontsize=9)

plt.suptitle('🏦 Train vs. Test Performance — Overfitting Diagnostic',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Verdict ───────────────────────────────────────────────────────────────────
print('\n=== Overfitting Verdict ===')
for model_name, train_s, test_s in [
    ('Random Forest', rf_train_scores,  rf_test_scores),
    ('XGBoost',       xgb_train_scores, xgb_test_scores)
]:
    f1_gap = train_s['F1 Score'] - test_s['F1 Score']
    if f1_gap > 0.05:
        status = '⚠️  Likely OVERFITTING — consider max_depth limit, more regularisation, or pruning'
    elif f1_gap > 0.02:
        status = '🟡 Mild overfit — monitor; may be acceptable'
    elif test_s['F1 Score'] < 0.75:
        status = '🔵 Possible UNDERFIT — both train and test scores are low'
    else:
        status = '✅ Well-generalised — small gap, strong test performance'
    print(f'{model_name:<16}  F1 gap = {f1_gap:+.4f}  →  {status}')

## 20a. Export Model for Console Predictor

> Run this cell once to save the trained XGBoost model.  
> Place `xgb_model.pkl` in the same folder as `loan_predictor.py`.

In [ ]:
import joblib

joblib.dump(xgb_model, 'xgb_model.pkl')
print('✅ XGBoost model saved to xgb_model.pkl')
print('   Copy this file to the same folder as loan_predictor.py')
print(f'   Model type: {type(xgb_model).__name__}')
print(f'   Best params: {xgb_model.get_params()}')

## 20. Summary & Business Insight

**Dataset Balance:** ✅ Well-balanced (44.95% Rejected / 55.05% Approved) — no resampling required

**Preprocessing Issues Resolved:**
- ✅ **Outlier skew** — Log1p transformation applied to `annual_income`, `savings_assets`, `current_debt`, `loan_amount`
- ✅ **Multicollinearity** — VIF check performed on ratio features; `payment_to_income_ratio` removed as most redundant
- ✅ **Categorical encoding** — One-Hot Encoding fit on training data only (no leakage)
- ✅ **Stratified splits** — Preserves sub-group proportions (loan_intent, product_type) in train and test
- ✅ **Binary rare-event features** — `defaults_on_file`, `derogatory_marks` retained as strong discriminative signals (0% defaults in approved vs 11.9% in rejected)

**Key Feature Signals:**
- `defaults_on_file` — near-perfect separator (0% approved with defaults)
- `credit_score` — 64.5-point gap between approved (672.6) and rejected (608.1)
- `debt_to_income_ratio` — approved avg 0.240 vs rejected 0.342
- `loan_intent` — Debt Consolidation has the lowest approval rate (36.6%); Education the highest (67.5%)

**Model Recommendation:**
- Both models are strong candidates for this dataset
- **XGBoost** tends to edge ahead on ROC AUC and F1 for tabular financial data
- **Random Forest** is more interpretable and less sensitive to hyperparameter tuning
- Use XGBoost for production deployment; Random Forest as interpretable baseline

**Business Connection:**
- Industry: Retail Banking, Digital Lending, Fintech
- Problem: Manual loan reviews are slow, inconsistent, and costly
- Value: Automate first-pass decisions; escalate borderline cases to human reviewers
- Priority Metric: **Precision** (avoid approving high-risk customers) paired with **Recall** (avoid rejecting creditworthy applicants)
- Deployment signal: `defaults_on_file` + `credit_score` + `debt_to_income_ratio` form the core decision trio

In [ ]:
import pandas as pd
from IPython.display import display

metrics_display = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']

summary_df = pd.DataFrame({
    'Random Forest': [rf_scores[m]  for m in metrics_display],
    'XGBoost':       [xgb_scores[m] for m in metrics_display]
}, index=metrics_display)

print('=== Summary & Business Insight — Model Results ===')
print()
print(summary_df.round(4).to_string())
print()

# Winner per metric
print('--- Winner per Metric ---')
for m in metrics_display:
    rf_val  = rf_scores[m]
    xgb_val = xgb_scores[m]
    if rf_val > xgb_val:
        print(f'  {m:<12} Random Forest  ({rf_val:.4f} vs {xgb_val:.4f})')
    elif xgb_val > rf_val:
        print(f'  {m:<12} XGBoost        ({xgb_val:.4f} vs {rf_val:.4f})')
    else:
        print(f'  {m:<12} Tie            ({rf_val:.4f})')

# Styled display
display(summary_df.style
    .format('{:.4f}')
    .highlight_max(axis=1, color='#d4edda')
    .set_caption('🏦 Model Comparison — green = winner per metric')
    .set_table_styles([{'selector': 'caption',
                         'props': [('font-size', '13px'), ('font-weight', 'bold')]}])
)

In [ ]:
# Dynamic summary table — populates automatically after models are trained
metrics_all = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']

header = f"{'Metric':<12} {'Random Forest':>15} {'XGBoost':>10}"
divider = '-' * len(header)
print('=== Model Performance Summary ===')
print(header)
print(divider)
for m in metrics_all:
    rf_val  = rf_scores[m]
    xgb_val = xgb_scores[m]
    winner  = '← RF' if rf_val > xgb_val else ('← XGB' if xgb_val > rf_val else '  tie')
    print(f"{m:<12} {rf_val:>15.4f} {xgb_val:>10.4f}  {winner}")
print(divider)
overall_winner = 'Random Forest' if sum(rf_scores[m] > xgb_scores[m] for m in metrics_all) > 2 else 'XGBoost'
print(f'Overall edge:   {overall_winner}')